# SemDiD — Full Experiment Run on Colab T4

Generates SemDiD feedback variants for the canonical ASAP sample (Exploring Venus, 100 essays). Runs only the SemDiD path — the three baseline methods are produced locally on Brett's Mac. SemDiD outputs land in `/content/results/generations/<essay_id>__semdid.json`, which can be copied directly into the local `results/generations/` tree and re-evaluated locally.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Then upload `data/sample_venus_100.json` from the project's `data/` folder to `/content/` (drag-and-drop into the Files pane on the left).

**Expected runtime:** ~60–90 s per essay × 100 essays = ~2 hours. Colab free-tier sessions are ~12 hours so this fits, but a disconnect would lose in-flight work — we save per essay so a re-run with `SKIP_EXISTING=True` resumes from where we left off.

## 1. Confirm GPU is attached

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA device. Switch runtime to GPU (T4) and rerun.'
props = torch.cuda.get_device_properties(0)
print(f'Device:  {props.name}')
print(f'Memory:  {props.total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')

## 2. Install dependencies

~10 min. Same pinned trio as `smoke_test.ipynb`, plus uninstalls torchcodec (ABI-mismatched against pinned torch).

In [ ]:
%pip install -q \
    'vllm==0.7.3' \
    'transformers==4.49.0' \
    'tokenizers>=0.21,<0.22' \
    sentence-transformers accelerate

%pip uninstall -y torchcodec
print('Dependencies installed. IMPORTANT: restart runtime now — Runtime -> Restart session.')

### Restart the runtime before continuing

`Runtime -> Restart session`. Required after upgrading `transformers`.

In [ ]:
# Run AFTER restart. Confirms versions match the SemDiD authors' window.
import importlib.metadata as md
for pkg in ('vllm', 'transformers', 'tokenizers', 'sentence-transformers', 'torch'):
    try:
        print(f'{pkg:<22} {md.version(pkg)}')
    except md.PackageNotFoundError:
        print(f'{pkg:<22} NOT INSTALLED')

try:
    md.version('torchcodec')
    print('\nWARNING: torchcodec is still installed; uninstall it.')
except md.PackageNotFoundError:
    print('\ntorchcodec is uninstalled (good).')

from transformers.cache_utils import OffloadedCache  # should not raise
print('OffloadedCache import works — transformers is compatible.')

## 3. Clone the SemDiD fork

Pulls Brett's fork (which has the four patches: signature fix, fp16, list-coerce, beam-expansion collapse).

In [ ]:
import os
if not os.path.exists('/content/SemDiD'):
    !git clone --depth 1 https://github.com/bhutley/SemDiD.git /content/SemDiD
else:
    print('SemDiD already present, skipping clone.')
!ls /content/SemDiD/my_lm_eval/lm_eval/models/semantic_search.py

## 4. Apply the SemDiD monkey-patch

Loaded via `importlib` (NOT `sys.path.insert`), to avoid shadowing the real PyPI `gguf` package that VLLM imports.

In [ ]:
import importlib.util

SPEC_PATH = '/content/SemDiD/my_lm_eval/lm_eval/models/semantic_search.py'
spec = importlib.util.spec_from_file_location('semdid_patch', SPEC_PATH)
semdid_patch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(semdid_patch)

semdid_patch.patch_transformers_generation()
print('Monkey-patch applied.')

## 5. Upload the canonical sample

Drag `data/sample_venus_100.json` from your local project folder into the Files pane on the left, putting it at `/content/sample_venus_100.json`. The cell below verifies it landed.

In [ ]:
import json, os

SAMPLE_PATH = '/content/sample_venus_100.json'
if not os.path.exists(SAMPLE_PATH):
    # Fallback: trigger Colab's upload widget if the file isn't there yet.
    from google.colab import files
    print('Upload sample_venus_100.json now:')
    uploaded = files.upload()
    if 'sample_venus_100.json' in uploaded:
        os.rename('sample_venus_100.json', SAMPLE_PATH)

with open(SAMPLE_PATH) as f:
    essays = json.load(f)
print(f'Loaded {len(essays)} essays from {SAMPLE_PATH}')
print(f'First essay: id={essays[0]["essay_id"]}, score={essays[0]["scores"]["holistic"]}, n_words={essays[0]["meta"]["n_words"]}')

## 6. Load Qwen-0.5B-Instruct (HuggingFace side)

Same model and dtype as the local baseline runs. SemDiD will *also* load this internally via VLLM (~2 GB total in fp16; fits T4 comfortably).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import torch

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16,  # T4 lacks bf16; matches local pipeline
    trust_remote_code=True,
)
model.use_semantic_diverse_beam_search = True   # the SemDiD patch checks this flag
print(f'Loaded {MODEL_NAME} on {model.device}')

## 7. Helpers — prompt + generation

Inlined from `src/feedback_pipeline.py` so this notebook is self-contained. **The prompt template is identical to the local pipeline** — that's what makes the cross-method comparison fair.

In [ ]:
import time
from pathlib import Path

FEEDBACK_SYSTEM_PROMPT = (
    'You are an experienced writing teacher providing constructive, specific, '
    'actionable feedback to a student on their essay.'
)

FOCUS_DIMENSIONS = ['argument development', 'evidence and support', 'organisation', 'mechanics']

def build_messages(essay_text):
    user_msg = (
        'Please give feedback on the following student essay. '
        f"Comment on {', '.join(FOCUS_DIMENSIONS)}. "
        'Write a single concise feedback paragraph (4-6 sentences).\n\n'
        f'Essay:\n{essay_text}\n\nFeedback:'
    )
    return [
        {'role': 'system', 'content': FEEDBACK_SYSTEM_PROMPT},
        {'role': 'user',   'content': user_msg},
    ]

def generate_semdid(essay_text, k=3, max_new_tokens=300):
    messages = build_messages(essay_text)
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(prompt_text, return_tensors='pt').input_ids.to(model.device)

    gen_config = GenerationConfig(
        max_length=input_ids.shape[1] + max_new_tokens,
        num_beams=k * 3,
        num_beam_groups=k,
        diversity_penalty=1.0,
        do_sample=False,
        repetition_penalty=1.2,
        forward_steps=300,
        num_return_sequences=k,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    start = time.time()
    with torch.inference_mode():
        outputs = model.generate(input_ids, generation_config=gen_config)
    elapsed = time.time() - start

    # SemDiD's patched path returns a list of decoded strings; stock HF
    # would return a (k, seq_len) tensor. Handle both.
    if isinstance(outputs, list) and all(isinstance(x, str) for x in outputs):
        variants = [s.strip() for s in outputs]
    else:
        variants = [
            tokenizer.decode(out[input_ids.shape[1]:], skip_special_tokens=True).strip()
            for out in outputs
        ]
    return variants, elapsed, prompt_text, gen_config.to_dict()

print('Helpers defined.')

## 8. Run over all essays

Saves one JSON per essay so a session disconnect doesn't lose work. Re-running this cell with `SKIP_EXISTING=True` resumes — already-saved essays are skipped.

Set `N_ESSAYS=5` for an initial smoke pass; bump to `100` for the full run once that works.

**Output layout matches the local pipeline:** `/content/results/generations/<essay_id>__semdid.json` mirrors `results/generations/<essay_id>__semdid.json` on your Mac, so the files drop straight into the same tree.

In [ ]:
from tqdm.auto import tqdm

# ---- Run config — change these as needed ----
N_ESSAYS = 5            # set 100 for the full run
SKIP_EXISTING = True    # skip essays already on disk
K = 3
MAX_NEW_TOKENS = 300
OUT_DIR = Path('/content/results/generations')
OUT_DIR.mkdir(parents=True, exist_ok=True)

to_process = essays[:N_ESSAYS]
print(f'Processing {len(to_process)} essays. Skip-existing: {SKIP_EXISTING}')

errors = []
for essay in tqdm(to_process):
    essay_id = essay['essay_id']
    target = OUT_DIR / f'{essay_id}__semdid.json'

    if SKIP_EXISTING and target.exists():
        continue

    try:
        variants, elapsed, prompt_text, config_dict = generate_semdid(
            essay['essay'], k=K, max_new_tokens=MAX_NEW_TOKENS,
        )
        result = {
            'essay_id': essay_id,
            'mode': 'semdid',
            'model_name': MODEL_NAME,
            'variants': variants,
            'k': K,
            'latency_s': elapsed,
            'prompt': prompt_text,
            'config': config_dict,
        }
        target.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    except Exception as e:
        errors.append((essay_id, type(e).__name__, str(e)))
        print(f'  [ERROR] {essay_id}: {type(e).__name__}: {e}')

saved = sorted(OUT_DIR.glob('*__semdid.json'))
print(f'\nDone. Saved {len(saved)} essays total.')
if errors:
    print(f'{len(errors)} errors:')
    for eid, etype, emsg in errors:
        print(f'  {eid}: {etype}: {emsg[:120]}')

## 9. Quick diversity sanity check

Compute mean pairwise cosine similarity across saved essays, just to confirm SemDiD is producing diverse output (we expect ~0.55–0.70 mean off-diagonal similarity, well below 0.85). Same metric the local pipeline uses.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

enc = SentenceTransformer('all-MiniLM-L6-v2')

per_essay = []
for path in sorted(OUT_DIR.glob('*__semdid.json')):
    rec = json.loads(path.read_text())
    if len(rec['variants']) < 2:
        continue
    embs = enc.encode(rec['variants'], convert_to_numpy=True, normalize_embeddings=True)
    sim = embs @ embs.T
    n = len(rec['variants'])
    off = (sim.sum() - n) / (n * (n - 1))
    per_essay.append((rec['essay_id'], float(off)))

if per_essay:
    sims = [s for _, s in per_essay]
    print(f'Essays:                  {len(sims)}')
    print(f'Mean similarity (overall): {np.mean(sims):.3f}')
    print(f'Median:                    {np.median(sims):.3f}')
    print(f'Min / max:                 {min(sims):.3f} / {max(sims):.3f}')
    print('\nFirst 10 essays:')
    for eid, s in per_essay[:10]:
        print(f'  {eid}  sim={s:.3f}')
else:
    print('No multi-variant generations found yet.')

## 10. Package and download results

Zips the per-essay JSONs and triggers a browser download. Unzip into your local `results/generations/` and re-run `python -m src.run_local --n 100 --skip-existing` locally to evaluate (the `--skip-existing` will keep your baseline generations and only run eval over the new SemDiD outputs).

In [ ]:
import shutil
from datetime import datetime

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
archive = f'/content/semdid_generations_{stamp}'
shutil.make_archive(archive, 'zip', root_dir='/content/results', base_dir='generations')
archive_path = archive + '.zip'
print(f'Created {archive_path} ({os.path.getsize(archive_path) / 1024:.1f} KB)')

from google.colab import files
files.download(archive_path)

## After downloading

On your Mac:
```bash
cd "NLP Group Project"
unzip ~/Downloads/semdid_generations_*.zip -d results/
ls results/generations/*__semdid.json | wc -l   # should match number of essays
python -m src.run_local --n 100 --skip-existing
```

The `--skip-existing` flag means generation gets reused (both baselines and the new SemDiD outputs). Eval will run fresh over all four methods and produce the combined `results/summary.json` with rows for greedy / temperature / diverse_beam / semdid — the input to the report's results section.